In [1]:
# This notebook aims to calculate the area of orthographic projections of our sample (in the .stl file) for varying sample angles, ψ.
# This is accomplished using a Monte Carlo approach.
# The projection is bounded and random coordinates generated within the area.
# The ratio of points in the sample projection to total is the ratio of areas.

using FileIO
using GeometryBasics
using MeshIO
using BenchmarkTools
include("Modules/projections.jl")
using .projections

In [2]:
# Retrieving the vertices and indices of the triangular mesh of the sample surface from the .stl file.


# Dummy sample comprising an icosphere with 320 faces.
stl = load("STL_FileExamples/Icosphere1280.stl")
vertices = GeometryBasics.coordinates(stl)
indices = GeometryBasics.faces(stl)
# Extracting the number of triangular faces used in the mesh.
const n_faces = length(indices)

1280

In [3]:
# Benchmarking the rotate! function used to rotate the crystal as determined by ψ.

@benchmark projections.rotate!(30f0, vertices)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  136.600 μs …  30.002 ms  ┊ GC (min … max):  0.00% … 0.00%
 Time  (median):     172.000 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   258.349 μs ± 584.085 μs  ┊ GC (mean ± σ):  13.64% ± 7.98%

  ▄▆█▇▅▄▃▃▄▄▅▄▃▃▂▂▂▁▁▁▂▂▃▃▂▂▂▂▂▂▂                               ▂
  █████████████████████████████████████▇▆▆▆▅▅▆▆▄▅▅▆▃▄▃▃▁▃▁▄▄▃▄▄ █
  137 μs        Histogram: log(frequency) by time        744 μs <

 Memory estimate: 301.16 KiB, allocs estimate: 7713.

In [4]:
# Projecting the sample onto the y-z plane (the plane perpendicular to the neutron beam).
# This is accomplished by simply reading off the y and z coordinates of the vertices.
# Grouping these projected vertices by the triangles they create.

p_vertices, p_triangles = projections.project(vertices, indices, n_faces)

(StaticArraysCore.SVector{2, Float32}[[0.9876975, 0.0067883935], [0.96487916, -0.122859836], [0.9510131, 0.009507999], [-0.8908923, 0.0067883935], [-0.91708285, -0.122859836], [-0.95101315, 0.009507999], [0.9876974, -0.0067883935], [0.96487916, 0.122859836], [0.95101315, -0.009507999], [0.076358944, 0.40318057]  …  [0.7953284, 0.54833657], [0.7544285, 0.63534635], [0.8463726, 0.5197157], [0.80901694, 0.5257311], [0.7841598, 0.62033826], [0.7544285, 0.63534635], [0.683161, 0.7278262], [0.7544285, 0.63534635], [0.7841598, 0.62033826], [0.8463726, 0.5197157]], Vector{StaticArraysCore.SVector{2, Float32}}[[[0.9876975, 0.0067883935], [0.96487916, -0.122859836], [0.9510131, 0.009507999]], [[-0.8908923, 0.0067883935], [-0.91708285, -0.122859836], [-0.95101315, 0.009507999]], [[0.9876974, -0.0067883935], [0.96487916, 0.122859836], [0.95101315, -0.009507999]], [[0.076358944, 0.40318057], [0.22658148, 0.39419597], [0.15436462, 0.2721286]], [[-0.5977941, 0.40318057], [-0.7068232, 0.39419597], [-0

In [5]:
# Finding the maximum and minimum value of each projected coordinate.
# This is used to create a 2D bounding rectangle and find it's area.

min_coord, max_coord, rec_area = projections.aabb_2d(p_vertices)

(Float32[-1.0, -1.0], Float32[1.0, 1.0], 4.0f0)

In [6]:
# Setting the total number of random coordinates to be generated within the bounding rectangle.

const n_tot = 100000

100000

In [7]:
# Testing the area calculation with an approximate circle of radius 1.
# Should act as a π approximator.

projections.area_calc(min_coord, max_coord, p_triangles, n_tot, n_faces, rec_area)

3.12052f0

In [8]:
# Benchmarking this area function.

@benchmark projections.area_calc(min_coord, max_coord, p_triangles, n_tot, n_faces, rec_area)

BenchmarkTools.Trial: 7 samples with 1 evaluation per sample.
 Range (min … max):  802.845 ms … 879.217 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     812.629 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   829.347 ms ±  29.760 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ██    ██                    █              █                █  
  ██▁▁▁▁██▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  803 ms           Histogram: frequency by time          879 ms <

 Memory estimate: 192 bytes, allocs estimate: 6.